# DS 2002 Midterm Project

Libraries

In [444]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sqlite3 as sql
import datetime as dt
import plotly.express as px
import plotly.graph_objects as go

Raw Data Files

In [445]:
# CSV Files
store_locations = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/store_locations.csv'
transactions = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/walmart_transactions.csv'
distribution_center = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/distribution_centers.csv'
product_catalog = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/product_catalog.csv'
# Database File
!wget -O walmart.db "https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/inventory_and_sales.db"
conn = sql.connect("walmart.db")


# Tables from CSV Files
loc = pd.read_csv(store_locations)
trans = pd.read_csv(transactions)
dist = pd.read_csv(distribution_center)
cat = pd.read_csv(product_catalog)

# Tables from database file
inv_levels_db = pd.read_sql("SELECT * FROM inventory_levels;", conn)
daily_sales_db = pd.read_sql("SELECT * FROM daily_sales_summary;", conn)

--2026-10-21 17:31:54--  https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/inventory_and_sales.db
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 352256 (344K) [application/octet-stream]
Saving to: ‘walmart.db’

walmart.db          100%[===================>] 344.00K  --.-KB/s    in 0.03s   

2026-10-21 17:31:54 (11.2 MB/s) - ‘walmart.db’ saved [352256/352256]



Cleaning

In [446]:
# Profile on the tables
def profile(df, name=None):
    """Return a summary of dtypes, null counts, null percentages, and unique counts."""
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_count": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
    })
    if name:
        print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return summary

tables = {"loc": loc, "trans": trans, "dist": dist, "cat": cat, "inv_levels_db":inv_levels_db, "daily_sales_db": daily_sales_db}

for name, df in tables.items():
    display(profile(df, name))
    print()


loc: 14 rows x 8 columns


,dtype,null_count,null_pct,unique
store_id,object,0,0.00,14
store_name,object,0,0.00,12
city,object,0,0.00,12
state,object,0,0.00,3
zip_code,int64,0,0.00,12
latitude,float64,1,7.14,11
longitude,float64,1,7.14,11
region,object,0,0.00,7



trans: 25,562 rows x 8 columns


,dtype,null_count,null_pct,unique
transaction_id,object,0,0.00,25532
store_id,object,0,0.00,60
timestamp,object,886,3.47,23952
sku,object,0,0.00,45
product_name,object,0,0.00,63
category,object,0,0.00,37
quantity,int64,0,0.00,11
unit_price,object,0,0.00,3668



dist: 5 rows x 8 columns


...

,dtype,null_count,null_pct,unique
sku,object,0,0.00,45
product_name,object,0,0.00,38
category,object,0,0.00,27
subcategory,object,13,28.89,19
unit_cost,float64,5,11.11,40
supplier,object,0,0.00,14
package_size,object,0,0.00,15



inv_levels_db: 4,320 rows x 6 columns


,dtype,null_count,null_pct,unique
date,object,0,0.00,30
store_id,object,0,0.00,12
sku,object,0,0.00,49
units_on_hand,int64,0,0.00,196
units_on_order,float64,626,14.49,101
reorder_point,int64,0,0.00,31



daily_sales_db: 3,600 rows x 5 columns


,dtype,null_count,null_pct,unique
date,object,0,0.0,60
store_id,object,0,0.0,12
category,object,0,0.0,10
total_units,float64,0,0.0,285
total_revenue,float64,0,0.0,3552


In [447]:
# Store Location Cleaning

# Normalizing spelling of Florida to FLORIDA
loc["state"] = (
    loc["state"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(".","", regex=False)
)
state_map = {
    "FLORIDA":"FLORIDA",
    "FLA":"FLORIDA",
    "FL":"FLORIDA",
}
loc["state"]=loc["state"].replace(state_map)

# Fixing store_id to drop duplicates
id_fix = {
    "FL239":"FL-239",
    "FL_287":"FL-287",
}
loc["store_id"]=loc["store_id"].replace(id_fix)
loc = loc.drop_duplicates(subset="store_id", keep="first").reset_index(drop=True)

In [448]:
# Distribution Cleaning

# Removing "~" and converting "surge_capacity" to Int64
dist["surge_capacity"] = (
    pd.to_numeric(dist["surge_capacity"].astype(str).str.replace("~", "", regex=False), errors="coerce")
    .astype("Int64")
)

# Need to make a decision on what to do for missing cost/shipment for DC-BHM

In [449]:
# Product Catalog Cleaning and SKU Mapping

# Cleaning category by normalizing case and consolidating category labels
cat["category"] = (
    cat["category"].str.strip().str.lower()
    .str.title()
)
category_map = {
        "Breakfast Foods": "Breakfast",
        "Emergency Supplies": "Emergency",
        "Alcoholic Beverages": "Alcohol",
        "Groceries": "Grocery",
        "Baby": "Baby Products",
}
cat["category"]=cat["category"].replace(category_map)

# SKU mapping. SKUs will be consolidated based on product name, ignoring all other columns such as cost, supplier and package size
sku_map = {
        'PT-12':'POPTART-STRAW',
        'SKU#459812':'POPTART-STRAW',
        'POPTART-STRAW':'POPTART-STRAW',
        'PT_STRAWBERRY':'POPTART-STRAW',
        'WTR-24':'WATER24PK',
        'WATER24PK':'WATER24PK',
        'WTR_24PK':'WATER24PK',
        'BATT-100': 'BAT-AA4',
        'BAT-AA4':'BAT-AA4',
        'BATT_AA_4PK':'BAT-AA4',
        'FLASH-01': 'LED-FLASH',
        'LED-FLASH': 'LED-FLASH',
        'BEER-24': 'BEER-24',
        'LTBEER24': 'BEER-24',
        'BG-MONO': 'GAME-MONO',
        'MONOPOLY-01': 'GAME-MONO',
        'GAME-MONO': 'GAME-MONO',
        'CARDS-01':'CARDS-01',
        'PLAYCARD-STD':'CARDS-01',
        'STREAM-01':'STREAM-01',
        'ROKU-STD':'STREAM-01',
        'SOUP-CAN':'SOUP',
        'CNSOUP-01':'SOUP',
        'BREAD-WH': 'WHITEBREAD',
        'WHITEBREAD': 'WHITEBREAD',
        'PB-16': "PNUTBTR",
        'PNUTBTR-16': "PNUTBTR",
        'CNDL-EM': "EMERG-CNDL",
        'CANDLE-EMRG': "EMERG-CNDL",
        'FAID-01': 'FIRSTAID-KIT',
        'FIRSTAID-KIT': 'FIRSTAID-KIT',
        'CHRG-PORT': 'CHRGR',
        'PORTCHARGER': 'CHRGR',
        'CHIPS-VP': "CHIPS-VP",
        'CHIPVARIETY': "CHIPS-VP",
        'PTOWEL-6': 'PTOWEL',
        'PAPERTOWEL6': 'PTOWEL',
        'TP-12': 'TOILETPAPER',
        'TOILETPAPER12': 'TOILETPAPER',
        'DIAP-S3':'DIAPERS',
        'DIAPERS-3':'DIAPERS',
        'BFORM-01':'BABYFORM',
        'BABYFORMULA':'BABYFORM',
        'TARP-HD': 'TARP-HD',
        'HDTARP-01': 'TARP-HD',
}
cat["sku"]=cat["sku"].replace(sku_map)

# Normalizing product_name. This also has functionality for the transaction data
groups = {
    "Pop-Tarts Strawberry":   ["pop-tarts strawberry", "Strawberry Pop-Tart", 'Pop Tarts Strawberry',"Pop-Tarts Strawberry",'POPTART-STRAWBERRY'],
    "Bottled Water 24-Pack":  ["BOTTLED WATER 24PK", "Bottled Water 24-Pack", "Water - 24 Pack"],
    "AA Batteries 4-Pack":    ["BATTERIES AA 4PK", "AA Batteries 4-Pack",'AA Batteries (4-pack)'],
    "LED Flashlight":         ["FLASHLIGHT LED", "LED Flashlight",'Flashlight - LED'],
    "Light Beer 24-Pack":     ["Light Beer (24-pack)", "Light Beer 24-Pack",'Light Beer 24pk','BEER LIGHT 24PK'],
    "Roku Streaming Stick":   ["Roku Streaming Stick", "Streaming Device",'STREAMING STICK'],
    "Monopoly Board Game":    ["Monopoly Board Game", "Board Game - Monopoly",'MONOPOLY'],
    "Playing Cards":          ["Playing Cards", "PLAYING CARDS",'Standard Playing Cards'],
    "Canned Soup":            ["Soup - Canned", "Canned Soup",'CANNED SOUP'],
    "White Bread":            ["WHITE BREAD", "White Bread",'Bread - White'],
    "Peanut Butter":          ["PEANUT BUTTER", "Peanut Butter",'Peanut Butter 16oz'],
    "Emergency Candles":      ["Candles (Emergency)", "Emergency Candles",'CANDLES - EMERGENCY'],
    "First Aid Kit":          ["First-Aid Kit", "First Aid Kit",'FIRST AID KIT'],
    "Portable Phone Charger": ["PORTABLE CHARGER", "Portable Phone Charger",'Phone Charger - Portable'],
    "Chips Variety Pack":     ["Variety Pack - Chips", "Chips Variety Pack",'CHIPS VARIETY PACK'],
    "Paper Towels":           ["Paper Towels",  'PAPER TOWELS 6PK','Paper Towels 6-Roll'],
    "Toilet Paper 12-Roll":   ["TOILET PAPER 12PK", "Toilet Paper 12-Roll",'TP 12 Pack'],
    "Baby Diapers - Size 3":  ["Baby Diapers - Size 3", "DIAPERS SZ 3",'Diapers Size 3'],
    "Baby Formula":           ["Infant Formula", "Baby Formula", 'BABY FORMULA'],
    "Heavy Duty Tarp":        ["Heavy Duty Tarp", "TARP - HEAVY DUTY",'HD Tarp'],
}


name_map = {variant: canon for canon, variants in groups.items() for variant in variants}
cat["product_name"] = cat["product_name"].str.strip().map(name_map)

# Converting package_size to Int64. For the entries with <NA>, they were formerly "bundle", which is not clear on how large the package size is.
size = cat["package_size"].str.strip().str.lower().replace({"single": "1", "bundle": pd.NA})
cat["package_size"] = size.str.extract(r"(\d+)", expand=False).astype("Int64")
cat.isna().sum()

# Need to decide on what to do with NaN in the "unit_cost" column


,0
sku,0
product_name,0
category,0
subcategory,13
unit_cost,5
supplier,0
package_size,3


In [450]:
trans
(trans["product_name"].unique())
# paper towels 6 pack

array(['PEANUT BUTTER', 'CANNED SOUP', 'Toilet Paper 12-Roll',
       'Roku Streaming Stick', 'Light Beer 24pk', 'Chips Variety Pack',
       'TARP - HEAVY DUTY', 'PAPER TOWELS 6PK', 'Emergency Candles',
       'PLAYING CARDS', 'CHIPS VARIETY PACK', 'Bread - White',
       'Portable Phone Charger', 'Paper Towels', 'FIRST AID KIT',
       'Peanut Butter', 'pop-tarts strawberry', 'Flashlight - LED',
       'Peanut Butter 16oz', 'POPTART-STRAWBERRY',
       'AA Batteries (4-pack)', 'Strawberry Pop-Tart', 'Infant Formula',
       'Water - 24 Pack', 'TOILET PAPER 12PK', 'STREAMING STICK',
       'BATTERIES AA 4PK', 'Standard Playing Cards', 'FLASHLIGHT LED',
       'Pop Tarts - Strawberry', 'PORTABLE CHARGER', 'DIAPERS SZ 3',
       'Diapers Size 3', 'Heavy Duty Tarp', 'White Bread',
       'Playing Cards', 'BOTTLED WATER 24PK', 'Soup - Canned',
       'Phone Charger - Portable', 'Monopoly Board Game',
       'Light Beer (24-pack)', 'BEER LIGHT 24PK', 'Variety Pack - Chips',
       'Candles

In [456]:
# Cleaning category by normalizing case and consolidating category labels
trans["category"] = (
    trans["category"].str.strip().str.lower()
    .str.title()
)
trans["category"]=trans["category"].replace(category_map)


# SKU map
trans["sku"]=trans["sku"].replace(sku_map)


# product names

print(trans["product_name"].unique())
print(trans["sku"].unique())
print(cat["sku"].unique())
trans.isna().sum()

['PEANUT BUTTER' 'CANNED SOUP' 'Toilet Paper 12-Roll'
 'Roku Streaming Stick' 'Light Beer 24pk' 'Chips Variety Pack'
 'TARP - HEAVY DUTY' 'PAPER TOWELS 6PK' 'Emergency Candles'
 'PLAYING CARDS' 'CHIPS VARIETY PACK' 'Bread - White'
 'Portable Phone Charger' 'Paper Towels' 'FIRST AID KIT' 'Peanut Butter'
 'pop-tarts strawberry' 'Flashlight - LED' 'Peanut Butter 16oz'
 'POPTART-STRAWBERRY' 'AA Batteries (4-pack)' 'Strawberry Pop-Tart'
 'Infant Formula' 'Water - 24 Pack' 'TOILET PAPER 12PK' 'STREAMING STICK'
 'BATTERIES AA 4PK' 'Standard Playing Cards' 'FLASHLIGHT LED'
 'Pop Tarts - Strawberry' 'PORTABLE CHARGER' 'DIAPERS SZ 3'
 'Diapers Size 3' 'Heavy Duty Tarp' 'White Bread' 'Playing Cards'
 'BOTTLED WATER 24PK' 'Soup - Canned' 'Phone Charger - Portable'
 'Monopoly Board Game' 'Light Beer (24-pack)' 'BEER LIGHT 24PK'
 'Variety Pack - Chips' 'Candles (Emergency)' 'CANDLES - EMERGENCY'
 'HD Tarp' 'Board Game - Monopoly' 'Streaming Device' 'LED Flashlight'
 'AA Batteries 4-Pack' 'Baby Diape

,0
transaction_id,0
store_id,0
timestamp,886
sku,0
product_name,0
category,0
quantity,0
unit_price,0


In [459]:
print(trans["sku"].unique())
print(cat["sku"].unique())

['PNUTBTR' 'SOUP' 'TOILETPAPER' 'STREAM-01' 'BEER-24' 'CHIPS-VP' 'TARP-HD'
 'PTOWEL' 'EMERG-CNDL' 'CARDS-01' 'WHITEBREAD' 'CHRGR' 'FIRSTAID-KIT'
 'POPTART-STRAW' 'LED-FLASH' 'BAT-AA4' 'BABYFORM' 'WATER24PK' 'DIAPERS'
 'GAME-MONO']
['POPTART-STRAW' 'WATER24PK' 'BAT-AA4' 'LED-FLASH' 'BEER-24' 'GAME-MONO'
 'CARDS-01' 'STREAM-01' 'SOUP' 'WHITEBREAD' 'PNUTBTR' 'EMERG-CNDL'
 'FIRSTAID-KIT' 'CHRGR' 'CHIPS-VP' 'PTOWEL' 'TOILETPAPER' 'DIAPERS'
 'BABYFORM' 'TARP-HD']


In [452]:
print(trans["store_id"].unique())

['FL-710' 'fl-412' 'FL-105' 'FL-330' 'FL-302' 'FL-445' 'FL_105' 'FL-619'
 ' FL-710' 'FL-412' 'FL-287' 'FL287' 'FL105' 'FL-239' 'fl-156' 'FL-156'
 'FL-501' 'FL-078' ' FL-330' 'fl-501' ' FL-105' 'FL412' 'FL_287' 'fl-239'
 'FL239' 'FL_078' 'fl-330' ' FL-287' 'fl-619' 'fl-078' 'FL_239' ' FL-302'
 'fl-105' ' FL-078' 'fl-287' 'FL_156' ' FL-239' ' FL-619' 'FL_445'
 'FL_501' 'FL_302' 'fl-710' 'FL330' 'FL445' 'FL_710' ' FL-156' 'FL_330'
 'FL_619' ' FL-412' 'FL302' 'FL078' 'fl-445' ' FL-445' 'FL619' 'FL710'
 'FL501' 'FL_412' 'fl-302' ' FL-501' 'FL156']


In [452]:
trans["product_name"] = trans["product_name"].str.strip().map(name_map)
